<a href="https://colab.research.google.com/github/Rohma1/personal-health-advisor/blob/main/Personal_Health_Advisor_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys

print("Python version:", sys.version)

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
!pip install -q transformers accelerate sentencepiece requests

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [4]:
SYSTEM_PROMPT = """
You are a Personalized Health Advisor, here to help users achieve their fitness and
nutrition goals with tailored workout routines, daily schedules, and meal plans. Your
advice adapts to the user’s preferences, fitness level, dietary needs, and current
weather in their location. Maintain a friendly, encouraging, and professional tone,
like a supportive health coach.

How to Engage Users:

- Start by asking for the user's location so you can check the weather.
- Encourage ongoing interaction and adaptation as the user's needs change.

Key Tasks:

1. Location and Weather:
Ask for the user's location to fetch current weather.
Adjust workouts based on weather conditions.

2. Workouts:
Ask about fitness goals, fitness level, preferred exercises, and health conditions.
Suggest routines tailored to their input and weather.

3. Meal Plans:
Ask about dietary preferences, restrictions, and nutritional goals.
Provide 2-3 daily meal options with approximate nutrition information.
Offer recipes when appropriate.

4. Flexibility:
Adjust advice based on user feedback.

5. Privacy:
Treat user information as private and use it only to personalize advice.

Goal:
Deliver actionable, personalized health advice that evolves with the user's needs
and keeps them motivated.
"""

In [5]:
def ask_llm(user_prompt):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        temperature=0.7
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response

In [6]:
response = ask_llm(
    "I want to build muscle. Give me a simple beginner workout."
)

print(response)

Great! Building muscle is all about strength training. Here’s a basic routine designed for beginners to start building your muscles safely and effectively:

### Warm-Up (5 minutes)
- **Jumping Jacks**: 10 rounds
- **Arm Circles**: 1 minute each direction, 6 circles per side
- **High Knees**: 1 minute

### Strength Training Routine (20 minutes)

#### Day 1: Chest & Triceps
- **Bench Press** (3 sets of 8 reps) – Use dumbbells or barbell if available.
- **Push-Ups** (3 sets of 8 reps) – If standard push-ups aren’t possible, do knee push-ups.
- **Dips** (3 sets of 8 reps) – On parallel bars.
- **Tricep Dips** (3 sets of 8 reps) – On a bench or chair.
- **Overhead Press** (3 sets of 8 reps) – Using dumbbells or a barbell.

#### Day 2: Back & Biceps
- **Pull-Ups** (3 sets of 8 reps) – If standard pull-ups aren't possible, use resistance bands or assisted pull-up machines.
- **Lat Pulldowns** (3 sets of 12 reps) – For back.
- **Bent Over Barbell Rows** (3 sets of 12 reps) – To work biceps and

In [7]:
import requests

def get_weather(latitude, longitude):

    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}"
        f"&longitude={longitude}"
        "&current=temperature_2m,relative_humidity_2m,"
        "precipitation,weather_code,wind_speed_10m"
    )

    response = requests.get(url)
    data = response.json()

    return data["current"]

In [8]:
weather = get_weather(24.8607, 67.0011)

print(weather)

{'time': '2026-09-17T10:00', 'interval': 900, 'temperature_2m': 30.6, 'relative_humidity_2m': 71, 'precipitation': 0.0, 'weather_code': 0, 'wind_speed_10m': 14.7}


In [9]:
locations = {
    # Pakistan
    "karachi": (24.8607, 67.0011),
    "lahore": (31.5204, 74.3587),
    "islamabad": (33.6844, 73.0479),
    "rawalpindi": (33.5651, 73.0169),
    "peshawar": (34.0151, 71.5249),
    "quetta": (30.1798, 66.9750),
    "multan": (30.1575, 71.5249),
    "faisalabad": (31.4504, 73.1350),
    "hyderabad": (25.3960, 68.3578),
    "sialkot": (32.4945, 74.5229),

    # Asia
    "delhi": (28.6139, 77.2090),
    "mumbai": (19.0760, 72.8777),
    "bangalore": (12.9716, 77.5946),
    "dhaka": (23.8103, 90.4125),
    "colombo": (6.9271, 79.8612),
    "kathmandu": (27.7172, 85.3240),
    "beijing": (39.9042, 116.4074),
    "shanghai": (31.2304, 121.4737),
    "tokyo": (35.6762, 139.6503),
    "seoul": (37.5665, 126.9780),
    "bangkok": (13.7563, 100.5018),
    "singapore": (1.3521, 103.8198),
    "jakarta": (-6.2088, 106.8456),
    "kuala_lumpur": (3.1390, 101.6869),
    "manila": (14.5995, 120.9842),
    "dubai": (25.2048, 55.2708),
    "riyadh": (24.7136, 46.6753),
    "doha": (25.2854, 51.5310),
    "tehran": (35.6892, 51.3890),
    "istanbul": (41.0082, 28.9784),

    # Europe
    "london": (51.5074, -0.1278),
    "paris": (48.8566, 2.3522),
    "berlin": (52.5200, 13.4050),
    "rome": (41.9028, 12.4964),
    "madrid": (40.4168, -3.7038),
    "barcelona": (41.3874, 2.1686),
    "amsterdam": (52.3676, 4.9041),
    "brussels": (50.8503, 4.3517),
    "vienna": (48.2082, 16.3738),
    "zurich": (47.3769, 8.5417),
    "moscow": (55.7558, 37.6173),
    "lisbon": (38.7223, -9.1393),
    "athens": (37.9838, 23.7275),
    "stockholm": (59.3293, 18.0686),

    # North America
    "new_york": (40.7128, -74.0060),
    "los_angeles": (34.0522, -118.2437),
    "chicago": (41.8781, -87.6298),
    "houston": (29.7604, -95.3698),
    "toronto": (43.6532, -79.3832),
    "vancouver": (49.2827, -123.1207),
    "mexico_city": (19.4326, -99.1332),
    "montreal": (45.5017, -73.5673),

    # South America
    "sao_paulo": (-23.5505, -46.6333),
    "rio_de_janeiro": (-22.9068, -43.1729),
    "buenos_aires": (-34.6037, -58.3816),
    "santiago": (-33.4489, -70.6693),
    "lima": (-12.0464, -77.0428),
    "bogota": (4.7110, -74.0721),

    # Africa
    "cairo": (30.0444, 31.2357),
    "lagos": (6.5244, 3.3792),
    "nairobi": (-1.2921, 36.8219),
    "cape_town": (-33.9249, 18.4241),
    "johannesburg": (-26.2041, 28.0473),
    "casablanca": (33.5731, -7.5898),

    # Oceania
    "sydney": (-33.8688, 151.2093),
    "melbourne": (-37.8136, 144.9631),
    "auckland": (-36.8509, 174.7645),
}

In [10]:
city = "karachi"

latitude, longitude = locations[city]

weather = get_weather(latitude, longitude)

print(weather)

{'time': '2026-09-17T10:00', 'interval': 900, 'temperature_2m': 30.6, 'relative_humidity_2m': 71, 'precipitation': 0.0, 'weather_code': 0, 'wind_speed_10m': 14.7}


In [11]:
def weather_summary(city):

    latitude, longitude = locations[city.lower()]

    weather = get_weather(latitude, longitude)

    return f"""
Location: {city}

Temperature: {weather['temperature_2m']}°C
Humidity: {weather['relative_humidity_2m']}%
Precipitation: {weather['precipitation']} mm
Wind speed: {weather['wind_speed_10m']} km/h
Weather code: {weather['weather_code']}
"""

In [12]:
print(weather_summary("Karachi"))


Location: Karachi

Temperature: 30.6°C
Humidity: 71%
Precipitation: 0.0 mm
Wind speed: 14.7 km/h
Weather code: 0



In [13]:
def search_web(query):

    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + query.replace(" ", "_")

    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        return data.get("extract", "No information found.")

    return "No information found."

In [14]:
print(search_web("Protein"))

No information found.


In [15]:
def health_agent(user_input, city=None):

    user_lower = user_input.lower()

    tool_results = ""

    if city and any(word in user_lower for word in [
        "weather",
        "rain",
        "raining",
        "outside",
        "outdoor",
        "today"
    ]):

        tool_results += weather_summary(city)

    prompt = f"""
User request:
{user_input}

Additional information from tools:
{tool_results}

Use the available information to provide a personalized response.
"""

    return ask_llm(prompt)

In [16]:
answer = health_agent(
    "It is raining today. Suggest a workout.",
    city="Karachi"
)

print(answer)

Hey there! It looks like it's going to be quite a rainy day here in Karachi. Don't worry; I've got some indoor activities just for you!

Given the wet weather, we could focus on some water-based workouts or try out yoga, which is great for flexibility and relaxation. How does that sound? Would you prefer something more active indoors or something calmer?

If you're up for an active session, maybe we can do a circuit training at home using things around your house. If not, let's look into some gentle yoga poses that will keep your body limber without getting too sweaty.

What would you like to do today?


In [17]:
def health_agent(user_input, city=None):

    user_lower = user_input.lower()

    tool_results = []

    if city and any(word in user_lower for word in [
        "weather",
        "rain",
        "raining",
        "temperature",
        "outside",
        "outdoor"
    ]):
        tool_results.append(
            "WEATHER INFORMATION:\n" +
            weather_summary(city)
        )

    if any(word in user_lower for word in [
        "nutrition",
        "protein",
        "food",
        "meal",
        "diet"
    ]):
        search_result = search_web("Nutrition")
        tool_results.append(
            "WEB INFORMATION:\n" +
            search_result
        )

    tools_text = "\n\n".join(tool_results)

    prompt = f"""
User request:
{user_input}

Information retrieved from tools:
{tools_text}

Use this information when relevant.
Do not mention internal tool implementation.
Give the user a clear, personalized response.
"""

    return ask_llm(prompt)

In [18]:
print(
    health_agent(
        "It's raining today. Suggest an indoor workout.",
        city="Karachi"
    )
)

Hey! It looks like it's going to be a bit of a downpour today. Don't worry though, there are plenty of fun indoor activities you can do instead!

Why don't we try out some yoga? Yoga is great for flexibility, stress relief, and staying active indoors. You can find many beginner-friendly yoga videos online or at your local gym if they offer virtual classes. This way, you get to move while enjoying the comfort of your home.

If you're feeling adventurous, you could also consider doing a short circuit training session using resistance bands or bodyweight exercises. Just make sure to stay hydrated and take breaks if needed. 

Remember, consistency is key when trying to maintain any fitness routine. So even if it's just a few minutes each day, give yourself credit for making progress!

Let me know what you think! Do you prefer more specific instructions or something else entirely?


In [19]:
print(
    health_agent(
        "What's a good meal plan for a vegetarian trying to build muscle?"
    )
)

Great question! To help you build muscle while sticking to your vegetarian diet, I'll suggest some meals that focus on protein-rich foods and nutrient-dense vegetables. Here are three meal options designed to meet your needs:

### Breakfast
**Option A:**
- **Eggs**: Cook two eggs sunny-side up or scramble them with spinach and mushrooms. Add a slice of whole-grain toast if desired.
- **Smoothie**: Blend together one cup of mixed berries (blueberries, strawberries), half a banana, a scoop of plant-based protein powder, and a splash of almond milk.

**Option B:**
- **Oatmeal**: Mix one cup of rolled oats with a tablespoon of chia seeds and a handful of almonds. Top with sliced bananas and a drizzle of honey.

### Lunch
**Option C:**
- **Quinoa Salad**: Combine cooked quinoa with diced cucumber, cherry tomatoes, red onion, avocado, and a simple dressing made with olive oil, lemon juice, and salt.
- **Stir-Fried Tofu**: Sauté tofu cubes with bell peppers, broccoli, and snap peas in a littl

In [20]:
print(
    health_agent(
        "I am a beginner and want to lose weight. I prefer simple home exercises.",
        city="Karachi"
    )
)

Great! Starting with simple home exercises is a good idea if you're new to exercise and looking to lose weight. Here’s an easy routine designed specifically for beginners like yourself:

### Monday: Full Body Workout
**Warm-up:** Jumping jacks (2 minutes)
**Exercises:**
1. **Push-Ups**: Do 8 sets of 10 repetitions.
2. **Plank**: Hold for 30 seconds, rest for 10 seconds between each set.
3. **Burpees**: Perform 6 reps, taking a deep breath after each rep.

**Cool Down:** Stretching (5 minutes)

This routine will help build strength, improve cardiovascular health, and burn calories effectively. Remember to listen to your body; if something feels too challenging, adjust the number of repetitions or sets accordingly.

If you have any specific questions or need more details, feel free to ask!

### Tuesday: Cardio & Strength Training
**Cardio:** 
- Walk briskly for 20 minutes at a steady pace.

**Strength Training:**
- **Bench Press**: Use dumbbells or resistance bands to lift weights agains

In [21]:
conversation = []

In [22]:
def chat(user_input, city=None):

    conversation.append({
        "role": "user",
        "content": user_input
    })

    context = "\n".join(
        f"{message['role']}: {message['content']}"
        for message in conversation
    )

    answer = health_agent(
        context,
        city
    )

    conversation.append({
        "role": "assistant",
        "content": answer
    })

    return answer

In [23]:
print(chat("I want to build muscle.", "Karachi"))

Great! Building muscle is a fantastic goal. To help you achieve this, let's focus on incorporating strength training into your routine. Here’s a basic plan that will challenge your muscles while allowing you to gradually increase weight over time:

**Warm-Up (10 minutes):**
Start with some dynamic stretches like arm circles, leg swings, and high knees. This prepares your body for more intense activity.

**Strength Training (30 minutes):**
- **Chest & Shoulders:** Focus on compound movements like bench press or shoulder presses. These will engage multiple muscle groups at once, helping you build overall strength.
- **Back & Biceps:** Include rows or pull-ups for back strengthening and hammer curls for bicep development.
- **Legs:** Incorporate squats, deadlifts, and lunges to target all major muscle groups in your legs.
- **Core:** Don’t forget to include planks, side planks, or Russian twists to strengthen your core.

**Cool Down (10 minutes):**
End with static stretching focusing on t

In [24]:
print(chat("I prefer vegetarian meals.", "Karachi"))

Thank you for letting me know about your preference for vegetarian meals! It sounds great to incorporate plant-based options into your diet. Let's tailor our meal plan around those choices.

### Vegetarian Meal Plan:

#### Breakfast:
- **Oatmeal with Fresh Berries**: Mix rolled oats with fresh berries (strawberries, blueberries), honey, and a drizzle of almond milk. Top with a sprinkle of chia seeds and nuts for added protein.
- **Smoothie Bowl**: Blend banana, spinach, almond milk, and a scoop of vanilla protein powder. Top with sliced bananas, strawberries, and granola.

#### Lunch:
- **Quinoa Salad**: Cook quinoa according to package instructions. Combine with mixed greens, cherry tomatoes, cucumber, avocado, and a light vinaigrette made with olive oil and lemon juice. Add grilled chicken breast slices for extra protein.

#### Snack:
- **Apple Slices with Almond Butter**: Cut an apple into wedges and top each slice with a tablespoon of almond butter for healthy fats and additional p

In [25]:
!pip install -q gradio

In [27]:
import gradio as gr

def chatbot(message, city):

    return chat(message, city)

interface = gr.Interface(
    fn=chatbot,
    inputs=[
        gr.Textbox(
            label="Your Question"
        ),
              gr.Dropdown(
          [
              "Karachi", "Lahore", "Islamabad", "Rawalpindi", "Peshawar",
              "Quetta", "Multan", "Faisalabad", "Hyderabad", "Sialkot",
              "Delhi", "Mumbai", "Bangalore", "Dhaka", "Colombo",
              "Kathmandu", "Beijing", "Shanghai", "Tokyo", "Seoul",
              "Bangkok", "Singapore", "Jakarta", "Kuala Lumpur", "Manila",
              "Dubai", "Riyadh", "Doha", "Tehran", "Istanbul",
              "London", "Paris", "Berlin", "Rome", "Madrid",
              "Barcelona", "Amsterdam", "Brussels", "Vienna", "Zurich",
              "Moscow", "Lisbon", "Athens", "Stockholm",
              "New York", "Los Angeles", "Chicago", "Houston", "Toronto",
              "Vancouver", "Mexico City", "Montreal",
              "São Paulo", "Rio de Janeiro", "Buenos Aires", "Santiago",
              "Lima", "Bogota",
              "Cairo", "Lagos", "Nairobi", "Cape Town", "Johannesburg",
              "Casablanca",
              "Sydney", "Melbourne", "Auckland"
          ],
          label="Location"
      )


    ],
    outputs=gr.Textbox(
        label="Personal Health Advisor"
    ),
    title="Personal Health Advisor",
    description="AI-powered fitness, nutrition and weather-aware health guidance"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://316bc9fc7f0b7fac6a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
